# Лабораторная работа 2 — JPEG-inspired compressor

Этот ноутбук демонстрирует работу реализованного компрессора. Все модули находятся в пакете `src/`.

**Запуск:**
1. `python -m src.test_data` — сгенерировать тестовые изображения
2. `python run_experiments.py` — получить сжатые файлы и графики
3. Этот ноутбук — интерактивная демонстрация каждого этапа

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path

%matplotlib inline

## Задание 1.1 — Тестовые данные и raw формат

In [ ]:
from src.test_data import generate_all
imgs = generate_all('images')

fig, axes = plt.subplots(1, 5, figsize=(18, 4))
for ax, (name, arr) in zip(axes, imgs.items()):
    if arr.ndim == 2:
        ax.imshow(arr, cmap='gray', vmin=0, vmax=255)
    else:
        ax.imshow(arr)
    ax.set_title(name)
    ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
from src.raw_format import save_raw, load_raw, raw_file_size, MODE_RGB, MODE_GRAY

# save in raw format
save_raw('output/demo.myrw', imgs['color'], MODE_RGB)

# read back
recovered, mode = load_raw('output/demo.myrw')
print(f"recovered shape = {recovered.shape}, identical: {np.array_equal(recovered, imgs['color'])}")

# compare sizes
raw = raw_file_size(imgs['color'], MODE_RGB)
png = Path('images/color_pattern.png').stat().st_size
print(f"raw = {raw} bytes, png = {png} bytes, png is {raw/png:.1f}x smaller")

## Задание 1.2 — Цветовые пространства RGB ↔ YCbCr

In [ ]:
from src.color_space import rgb_to_ycbcr, ycbcr_to_rgb

ycbcr = rgb_to_ycbcr(imgs['lena'])
back   = ycbcr_to_rgb(ycbcr)
err    = np.abs(back.astype(int) - imgs['lena'].astype(int)).max()

fig, axes = plt.subplots(1, 5, figsize=(18, 4))
axes[0].imshow(imgs['lena']);      axes[0].set_title('original RGB');    axes[0].axis('off')
axes[1].imshow(ycbcr[..., 0], cmap='gray'); axes[1].set_title('Y');     axes[1].axis('off')
axes[2].imshow(ycbcr[..., 1], cmap='gray'); axes[2].set_title('Cb');    axes[2].axis('off')
axes[3].imshow(ycbcr[..., 2], cmap='gray'); axes[3].set_title('Cr');    axes[3].axis('off')
axes[4].imshow(back);              axes[4].set_title(f'roundtrip (max err={err})'); axes[4].axis('off')
plt.tight_layout(); plt.show()

: 

## Задание 1.3 — Downsampling / Upsampling / Interpolation

In [ ]:
from src.resampling import downsample_2x, upsample_2x, resize_bilinear

# показать нарастающую пикселизацию при многократной децимации
img = imgs['lena']
d1 = downsample_2x(img)
d2 = downsample_2x(d1)
d3 = downsample_2x(d2)

# апсемплинг обратно до исходного размера
u3 = upsample_2x(upsample_2x(upsample_2x(d3)))
bil = resize_bilinear(d3, (img.shape[0], img.shape[1]))

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
axes[0].imshow(img);  axes[0].set_title(f'original {img.shape[:2]}');        axes[0].axis('off')
axes[1].imshow(d3);   axes[1].set_title(f'down x8 {d3.shape[:2]}');          axes[1].axis('off')
axes[2].imshow(u3);   axes[2].set_title('up x8 (nearest) — блочность');     axes[2].axis('off')
axes[3].imshow(bil);  axes[3].set_title('bilinear resize — гладко');         axes[3].axis('off')
plt.tight_layout(); plt.show()

## Задание 1.4 — DCT: прямое, обратное, матричное

In [ ]:
from src.dct import dct2_naive, idct2_naive, dct2_matrix, idct2_matrix

block = imgs['gray'][100:108, 100:108].astype(np.float32) - 128

c_naive  = dct2_naive(block)
c_matrix = dct2_matrix(block)
print(f'Naive vs matrix DCT — max abs diff: {np.abs(c_naive - c_matrix).max():.2e}')

rec = idct2_matrix(c_matrix)
print(f'DCT→IDCT reconstruction error: {np.abs(rec - block).max():.2e}')

fig, axes = plt.subplots(1, 3, figsize=(10, 3))
axes[0].imshow(block, cmap='gray'); axes[0].set_title('block')
axes[1].imshow(c_matrix, cmap='seismic', vmin=-200, vmax=200); axes[1].set_title('DCT coefficients')
axes[2].imshow(rec, cmap='gray');   axes[2].set_title('after IDCT')
for a in axes: a.axis('off')
plt.tight_layout(); plt.show()

### Временная сложность DCT
- Наивная формула: **O((NM)²)** — для блока 8×8 это 4096 умножений и 4096 косинусов (предвычислены).
- Матричная форма `S = Cᵀ s C`: **O(N³)** с меньшей константой и возможностью BLAS.
- Коэффициенты хранятся как `float32`/`float64` (они вещественные), после квантования — `int32`.

## Задание 1.4 — Квантование и Задание 2.6 — адаптация таблицы под quality

In [ ]:
from src.dct import STD_LUMA_Q, quality_scale_q, quantise, dequantise

print('Standard luminance Q table:')
print(STD_LUMA_Q)
print('\nQ(90) — высокое качество, малые значения:')
print(quality_scale_q(STD_LUMA_Q, 90))
print('\nQ(10) — низкое качество, большие значения:')
print(quality_scale_q(STD_LUMA_Q, 10))

## Задание 2.2 — Зигзаг обход

In [ ]:
from src.zigzag import zigzag, inverse_zigzag

m = np.arange(64).reshape(8, 8)
print('8x8 matrix zigzag order:')
print(zigzag(m))

# прямоугольная
m2 = np.arange(12).reshape(3, 4)
print('\n3x4 matrix:')
print(m2)
print('zigzag:', zigzag(m2))

## Задания 2.3–2.5 — Дифф. кодирование, RLE, Хаффман (полный цикл)

In [ ]:
from src.entropy import (differential_encode_dc, run_length_encode_ac,
                         vli_bits, vli_decode)

# 1) DC differential
dc = [128, 130, 125, 126, 140]
print('DC values:       ', dc)
print('DC differentials:', differential_encode_dc(dc))

# 2) AC RLE
ac = [3, 0, 0, -2, 0, 0, 0, 5] + [0]*55
print('\nAC input:         ', ac[:10], '...')
print('AC RLE triples:   ', run_length_encode_ac(ac))

# 3) VLI для категорий
for v in [-5, -1, 0, 1, 5, 100]:
    size, bits = vli_bits(v)
    print(f'  value={v:4d}  size={size}  bits=0b{bits:0{max(size,1)}b}  decoded={vli_decode(size, bits)}')

## Полный цикл: compress → decompress

In [ ]:
from src.compressor import compress, decompress

img = imgs['lena']
qualities = [10, 30, 50, 80]

fig, axes = plt.subplots(1, 5, figsize=(20, 4))
axes[0].imshow(img); axes[0].set_title('original'); axes[0].axis('off')

for i, q in enumerate(qualities, start=1):
    data = compress(img, quality=q)
    recovered = decompress(data)
    axes[i].imshow(recovered)
    axes[i].set_title(f'q={q}\n{len(data)} B')
    axes[i].axis('off')
plt.tight_layout(); plt.show()

## Графики размер-vs-качество (см. `output/size_vs_quality_*.png`)

In [ ]:
img = Image.open('output/size_vs_quality_all.png')
plt.figure(figsize=(10, 7))
plt.imshow(img); plt.axis('off'); plt.show()